# AML Benchmark — Part B PAI-HNU Run Notebook

**Part B primary strategy:** Part-A-Informed Hard-Negative Undersampling (**PAI-HNU**)  
**Goal:** Level-3 Mini-End-to-End-Smoke first, then full Part-B runs only after manual confirmation.

This notebook is intentionally staged:

1. Mount Drive and clone the Part-B branch.
2. Install dependencies and verify the sampler tests.
3. Copy/reuse Part-A artefacts from Drive.
4. Generate/cache baseline train scores.
5. Run **Mini-End-to-End-Smoke** with `--sample-n-train`.
6. Inspect smoke outputs.
7. Only after manual confirmation: run all three full PAI-HNU prevalences.
8. Generate Table 6 and back up results to Drive.

**Do not run the Full-Run cells until the smoke outputs are checked.**


## 0 — Configuration


In [ ]:
from pathlib import Path
import os, json, shutil, datetime, subprocess, sys

# === Edit only if your paths/branch changed ===
GITHUB_REPO = "https://github.com/fdrmic/classimbalance.git"
BRANCH = "feature/part-b-hard-negative-undersampling"

PROJECT_DIR = Path("/content/classimbalance")
DRIVE_ROOT = Path("/content/drive/MyDrive")
DRIVE_RESULTS = DRIVE_ROOT / "aml_results"

# Existing Part-A completed run on Drive
PART_A_RUN_DIR = DRIVE_RESULTS / "large_run_v2_20260407_1904"

# Existing Part-A XGBoost Baseline model used to score train rows for hard-negative mining
BASELINE_MODEL_PATH = PART_A_RUN_DIR / "runs" / "xgboost__baseline__p001__20260404_143052" / "model.pkl"

# PAI-HNU config inside the repo
PATHS_CONFIG = PROJECT_DIR / "configs" / "paths_large_part_b_pai_hnu.yaml"

# Smoke size. 200k is large enough to catch alignment/I/O issues but still cheap.
SMOKE_N_TRAIN = 200_000
SMOKE_PREVALENCE = "0.01"

print("PART_A_RUN_DIR      :", PART_A_RUN_DIR)
print("BASELINE_MODEL_PATH :", BASELINE_MODEL_PATH)
print("PROJECT_DIR         :", PROJECT_DIR)


## 1 — Mount Google Drive


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
print("Drive mounted.")


## 2 — Runtime check: RAM and GPU


In [ ]:
import psutil, os, subprocess, textwrap

ram = psutil.virtual_memory()
print(f"Total RAM     : {ram.total / 1e9:.1f} GB")
print(f"Available RAM : {ram.available / 1e9:.1f} GB")
print()

!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader 2>/dev/null || echo "No GPU detected"

if ram.total < 80e9:
    print("\nWARNING: Less than ~80 GB RAM detected. For full runs, use Colab Pro/Pro+ high-RAM runtime.")


## 3 — Clone repository branch


In [ ]:
import os, shutil
from pathlib import Path

if PROJECT_DIR.exists():
    print(f"Removing existing clone: {PROJECT_DIR}")
    shutil.rmtree(PROJECT_DIR)

cmd = ["git", "clone", "-b", BRANCH, GITHUB_REPO, str(PROJECT_DIR)]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)

os.chdir(PROJECT_DIR)
print("Working directory:", os.getcwd())
print("Branch:", subprocess.check_output(["git", "branch", "--show-current"], text=True).strip())
print("Latest commit:")
print(subprocess.check_output(["git", "--no-pager", "log", "--oneline", "-1"], text=True))


## 4 — Install dependencies


In [ ]:
import os, subprocess, sys
os.chdir(PROJECT_DIR)

# PyYAML is needed for config editing; pytest for Level-1 tests.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "pytest"], check=True)

# Project install
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)

# Requirements, if present
req = PROJECT_DIR / "requirements.txt"
if req.exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)], check=True)

print("Installation complete.")


## 5 — Verify imports and run Level-1 unit tests


In [ ]:
import os, subprocess, sys
os.chdir(PROJECT_DIR)

# Import check
from aml_benchmark.sampling.hard_negative_undersampling import build_pai_hnu_training_indices, validate_no_overlap
print("Import OK: PAI-HNU sampler available.")

# Level 1 tests
subprocess.run([sys.executable, "-m", "pytest", "tests/test_pai_hnu_sampler.py", "-v"], check=True)


## 6 — Level-2 sampler smoke test


In [ ]:
import numpy as np
from aml_benchmark.sampling.hard_negative_undersampling import build_pai_hnu_training_indices, validate_no_overlap

rng = np.random.default_rng(0)
n = 100_000
n_pos = 120

y = np.zeros(n, dtype=np.int8)
y[rng.choice(n, size=n_pos, replace=False)] = 1
scores = rng.uniform(size=n).astype(np.float32)

sel = build_pai_hnu_training_indices(
    y,
    scores,
    target_prevalence=0.005,
    random_state=42,
)

validate_no_overlap(sel.pos_idx, sel.hard_neg_idx, sel.temporal_neg_idx, sel.global_neg_idx)
print("OK -", sel.counts)


## 7 — Copy/reuse Part-A artefacts from Drive


In [ ]:
import shutil, os, json
from pathlib import Path

os.chdir(PROJECT_DIR)

if not PART_A_RUN_DIR.exists():
    raise FileNotFoundError(f"Part-A run folder not found: {PART_A_RUN_DIR}")

print("Using Part-A run folder:", PART_A_RUN_DIR)

# Destination folders expected by the project
splits_dst = PROJECT_DIR / "data" / "splits_v2"
processed_dst = PROJECT_DIR / "data" / "processed_v2"
runs_dst = PROJECT_DIR / "outputs" / "runs_v2"
leaderboard_dst = PROJECT_DIR / "outputs" / "leaderboard_v2"
results_dst = PROJECT_DIR / "results"

for p in [splits_dst, processed_dst, runs_dst, leaderboard_dst, results_dst]:
    p.mkdir(parents=True, exist_ok=True)

# Copy splits/features if present
for src_name, dst in [
    ("splits", splits_dst),
    ("splits_v2", splits_dst),
    ("processed", processed_dst),
    ("processed_v2", processed_dst),
    ("runs", runs_dst),
    ("runs_v2", runs_dst),
    ("leaderboard", leaderboard_dst),
    ("leaderboard_v2", leaderboard_dst),
]:
    src = PART_A_RUN_DIR / src_name
    if src.exists():
        print(f"Copying {src} -> {dst}")
        shutil.copytree(src, dst, dirs_exist_ok=True)

# Copy Part-A summary CSV if available in known locations.
summary_candidates = [
    PART_A_RUN_DIR / "part_a_summary_v2.csv",
    PART_A_RUN_DIR / "leaderboard" / "part_a_summary_v2.csv",
    PART_A_RUN_DIR / "leaderboard_v2" / "part_a_summary_v2.csv",
    PART_A_RUN_DIR / "outputs" / "leaderboard_v2" / "part_a_summary_v2.csv",
    DRIVE_RESULTS / "part_a_summary_v2.csv",
]
copied_summary = False
for cand in summary_candidates:
    if cand.exists():
        print("Copying Part-A summary:", cand)
        shutil.copy2(cand, results_dst / "part_a_summary_v2.csv")
        shutil.copy2(cand, leaderboard_dst / "part_a_summary_v2.csv")
        copied_summary = True
        break

if not copied_summary:
    print("WARNING: part_a_summary_v2.csv not found in known Drive locations.")
    print("Table 6 can be generated later after copying it to:")
    print(" -", results_dst / "part_a_summary_v2.csv")
    print(" -", leaderboard_dst / "part_a_summary_v2.csv")

print("\nSplits/features currently available:")
for f in sorted(splits_dst.glob("*"))[:40]:
    print(" -", f.relative_to(PROJECT_DIR), f.stat().st_size / 1e6, "MB")
print("...")

print("\nRuns available:")
for f in sorted(runs_dst.glob("xgboost__baseline__*"))[:10]:
    print(" -", f.relative_to(PROJECT_DIR))


## 8 — Configure `paths_large_part_b_pai_hnu.yaml` for Colab


In [ ]:
import yaml, os
from pathlib import Path

os.chdir(PROJECT_DIR)

if not PATHS_CONFIG.exists():
    raise FileNotFoundError(f"Missing config: {PATHS_CONFIG}")

with open(PATHS_CONFIG, "r") as f:
    cfg = yaml.safe_load(f) or {}

# Keep raw_dir optional; Part B should use cached features/splits, not raw reprocessing.
cfg["raw_dir"] = str(DRIVE_ROOT / "aml_data")
cfg["processed_dir"] = str(PROJECT_DIR / "data" / "processed_v2")
cfg["splits_dir"] = str(PROJECT_DIR / "data" / "splits_v2")

# Part-B final outputs are isolated from Part-A outputs.
cfg["outputs_dir"] = str(PROJECT_DIR / "outputs" / "runs_part_b_pai_hnu")
cfg["leaderboard_dir"] = str(PROJECT_DIR / "outputs" / "leaderboard_part_b_pai_hnu")

# Optional Part-A outputs only for reading/discovery. Never write Part-B results here.
cfg["part_a_outputs_dir"] = str(PROJECT_DIR / "outputs" / "runs_v2")

# Explicit model path can also be passed by CLI; keeping it here helps reproducibility.
cfg["baseline_model_path"] = str(BASELINE_MODEL_PATH)

with open(PATHS_CONFIG, "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print("Updated config:")
print(yaml.safe_dump(cfg, sort_keys=False))


## 9 — Sanity checks before scoring


In [ ]:
from pathlib import Path
import os, json

os.chdir(PROJECT_DIR)

print("Baseline model exists:", BASELINE_MODEL_PATH.exists(), BASELINE_MODEL_PATH)
if not BASELINE_MODEL_PATH.exists():
    raise FileNotFoundError(f"Baseline model not found: {BASELINE_MODEL_PATH}")

# Required split/feature files are validated by project code too; this gives an early visual check.
splits_dir = PROJECT_DIR / "data" / "splits_v2"
print("\nFiles in data/splits_v2:")
for pattern in ["*train*", "*val*", "*test*", "*features*"]:
    matches = sorted(splits_dir.glob(pattern))
    print(f"\nPattern {pattern}: {len(matches)} matches")
    for f in matches[:20]:
        print(" -", f.name, f.stat().st_size / 1e6, "MB")

print("\nConfig path:", PATHS_CONFIG)


## 10 — Level-3A: Generate/cache baseline train scores

This step uses the existing Part-A XGBoost Baseline model to score **training rows only**.  
Output should be `baseline_train_scores.parquet` with at least `row_idx` and `score`.

This can take time because it scores the full train set. It is still required before PAI-HNU can choose hard negatives.


In [ ]:
import os, subprocess, sys
from pathlib import Path

os.chdir(PROJECT_DIR)

score_cache = PROJECT_DIR / "data" / "splits_v2" / "baseline_train_scores.parquet"
score_meta = PROJECT_DIR / "data" / "splits_v2" / "baseline_train_scores_meta.json"

if score_cache.exists() and score_meta.exists():
    print("Score cache already exists. Skipping scoring.")
    print(" -", score_cache, score_cache.stat().st_size / 1e9, "GB")
    print(" -", score_meta)
else:
    cmd = [
        sys.executable, "-m", "aml_benchmark.experiments.score_baseline_train",
        "--paths", str(PATHS_CONFIG),
        "--baseline-model-path", str(BASELINE_MODEL_PATH),
    ]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)

print("\nScore cache status:")
print("parquet exists:", score_cache.exists(), score_cache)
print("meta exists   :", score_meta.exists(), score_meta)


## 11 — Inspect baseline score cache metadata


In [ ]:
import json
from pathlib import Path

score_meta = PROJECT_DIR / "data" / "splits_v2" / "baseline_train_scores_meta.json"
if score_meta.exists():
    meta = json.loads(score_meta.read_text())
    print(json.dumps(meta, indent=2)[:4000])
else:
    print("No meta file found.")

# Lightweight schema check, avoids loading all rows.
import pandas as pd
score_cache = PROJECT_DIR / "data" / "splits_v2" / "baseline_train_scores.parquet"
if score_cache.exists():
    df_head = pd.read_parquet(score_cache, columns=["row_idx", "score"]).head()
    print(df_head)
    print("Columns OK:", set(["row_idx", "score"]).issubset(df_head.columns))


## 12 — Level-3B: Mini-End-to-End Smoke Run

This is the important smoke test. It trains on a deterministic row-aligned subset and writes to:

`outputs/runs_part_b_pai_hnu_smoke/<run_id>/`

Expected metadata:
- `smoke_subsample_used: true`
- `sample_n_train: 200000`
- `row_index_mode: internal_0_based_with_orig_row_idx_mapping`
- `subsample_row_mapping.parquet` exists

Do **not** run full Part-B runs before this is checked.


In [ ]:
import os, subprocess, sys
os.chdir(PROJECT_DIR)

cmd = [
    sys.executable, "-m", "aml_benchmark.experiments.run_part_b_pai_hnu",
    "--paths", str(PATHS_CONFIG),
    "--target-prevalences", SMOKE_PREVALENCE,
    "--sample-n-train", str(SMOKE_N_TRAIN),
]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)


## 13 — Inspect latest Smoke output


In [ ]:
import json, os
from pathlib import Path
import pandas as pd

smoke_root = PROJECT_DIR / "outputs" / "runs_part_b_pai_hnu_smoke"
if not smoke_root.exists():
    raise FileNotFoundError(f"Smoke output root missing: {smoke_root}")

runs = sorted([p for p in smoke_root.iterdir() if p.is_dir()], key=lambda p: p.stat().st_mtime, reverse=True)
if not runs:
    raise FileNotFoundError(f"No smoke run folders found in {smoke_root}")

latest = runs[0]
print("Latest smoke run:", latest)

def show_json(name, max_chars=5000):
    p = latest / name
    print("\n" + "=" * 80)
    print(name, "exists:", p.exists())
    if p.exists():
        data = json.loads(p.read_text())
        print(json.dumps(data, indent=2)[:max_chars])
    return p

run_config_p = show_json("run_config.json")
manifest_p = show_json("sampling_manifest.json")
show_json("metrics_val_opt.json")
show_json("metrics_test_opt.json")
show_json("metrics_val.json")
show_json("metrics_test.json")

mapping = latest / "subsample_row_mapping.parquet"
print("\n" + "=" * 80)
print("subsample_row_mapping.parquet exists:", mapping.exists())
if mapping.exists():
    m = pd.read_parquet(mapping)
    print(m.head())
    print("Rows:", len(m))
    print("Columns:", list(m.columns))
    print("orig_row_idx monotonic:", m["orig_row_idx"].is_monotonic_increasing)
    print("internal_row_idx starts at 0:", int(m["internal_row_idx"].iloc[0]) == 0)


## 14 — Back up Smoke output to Drive


In [ ]:
import shutil, datetime
from pathlib import Path

ts = datetime.datetime.now().strftime("%Y%m%d_%H%M")
backup_dir = DRIVE_RESULTS / f"part_b_pai_hnu_smoke_{ts}"
backup_dir.mkdir(parents=True, exist_ok=True)

smoke_root = PROJECT_DIR / "outputs" / "runs_part_b_pai_hnu_smoke"
if smoke_root.exists():
    shutil.copytree(smoke_root, backup_dir / "runs_part_b_pai_hnu_smoke", dirs_exist_ok=True)
    print("Smoke outputs backed up to:", backup_dir)
else:
    print("Smoke root not found:", smoke_root)


---

# STOP

At this point, send the following files/contents for review before starting Full Runs:

- latest `sampling_manifest.json`
- latest `run_config.json`
- latest `metrics_val_opt.json`
- latest `metrics_test_opt.json`
- confirmation that `subsample_row_mapping.parquet` exists

Only continue after manual confirmation.


## 15 — Full Runs: manual confirmation guard


In [ ]:
# Change to True only after the smoke output has been reviewed.
RUN_FULL = False

if not RUN_FULL:
    raise RuntimeError("Full runs are blocked. Set RUN_FULL=True only after smoke output is approved.")


## 16 — Full Runs: run all three PAI-HNU prevalences

This cell runs the final three PAI-HNU experiments. It runs one prevalence at a time and backs up after each run.

Expected final output root:

`outputs/runs_part_b_pai_hnu/`


In [ ]:
import os, subprocess, sys, shutil, datetime
from pathlib import Path

os.chdir(PROJECT_DIR)

if not RUN_FULL:
    raise RuntimeError("RUN_FULL is False. Do not bypass this guard without manual approval.")

prevalences = ["0.001", "0.005", "0.01"]

for p in prevalences:
    print("\n" + "=" * 100)
    print(f"Running full PAI-HNU prevalence {p}")
    print("=" * 100)

    cmd = [
        sys.executable, "-m", "aml_benchmark.experiments.run_part_b_pai_hnu",
        "--paths", str(PATHS_CONFIG),
        "--target-prevalences", p,
    ]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)

    # Backup after each prevalence to reduce loss risk after disconnect.
    ts = datetime.datetime.now().strftime("%Y%m%d_%H%M")
    backup_dir = DRIVE_RESULTS / f"part_b_pai_hnu_incremental_{p.replace('.', '')}_{ts}"
    backup_dir.mkdir(parents=True, exist_ok=True)

    src = PROJECT_DIR / "outputs" / "runs_part_b_pai_hnu"
    if src.exists():
        shutil.copytree(src, backup_dir / "runs_part_b_pai_hnu", dirs_exist_ok=True)
        print("Incremental backup:", backup_dir)
    else:
        print("WARNING: Full output root not found:", src)


## 17 — Generate Table 6 after all three full runs


In [ ]:
import os, subprocess, sys
from pathlib import Path

os.chdir(PROJECT_DIR)

cmd = [sys.executable, "-m", "aml_benchmark.analysis.results_tables"]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)

print("\nCandidate result/table files:")
for root in [PROJECT_DIR / "results", PROJECT_DIR / "outputs", PROJECT_DIR / "outputs" / "leaderboard_part_b_pai_hnu"]:
    if root.exists():
        print("\nROOT:", root)
        for f in sorted(root.rglob("*table*"))[:50]:
            print(" -", f.relative_to(PROJECT_DIR))
        for f in sorted(root.rglob("*.csv"))[:50]:
            if "table" in f.name.lower() or "part" in f.name.lower():
                print(" -", f.relative_to(PROJECT_DIR))


## 18 — Final Drive backup


In [ ]:
import shutil, datetime
from pathlib import Path

ts = datetime.datetime.now().strftime("%Y%m%d_%H%M")
backup_dir = DRIVE_RESULTS / f"part_b_pai_hnu_run_{ts}"
backup_dir.mkdir(parents=True, exist_ok=True)

items = [
    PROJECT_DIR / "outputs" / "runs_part_b_pai_hnu",
    PROJECT_DIR / "outputs" / "leaderboard_part_b_pai_hnu",
    PROJECT_DIR / "results",
    PROJECT_DIR / "data" / "splits_v2" / "baseline_train_scores.parquet",
    PROJECT_DIR / "data" / "splits_v2" / "baseline_train_scores_meta.json",
]

for src in items:
    if not src.exists():
        print("Skip missing:", src)
        continue

    dst = backup_dir / src.relative_to(PROJECT_DIR)
    dst.parent.mkdir(parents=True, exist_ok=True)

    if src.is_dir():
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print("Copied dir :", src.relative_to(PROJECT_DIR), "->", dst)
    else:
        shutil.copy2(src, dst)
        print("Copied file:", src.relative_to(PROJECT_DIR), "->", dst)

print("\nFinal backup complete:", backup_dir)


## Notes

- `score_baseline_train.py` must only score training rows. Validation/test scores are not used for sampling.
- The smoke run uses internal `0..n_sub-1` indices after row-aligned subsampling and stores `orig_row_idx` in `subsample_row_mapping.parquet`.
- The full run operates on the full training row order.
- Table 6 intentionally requires all three full PAI-HNU prevalences. If one prevalence is missing, rerun only the missing prevalence.
- Back up after each full prevalence to avoid losing progress after Colab disconnects.
